# Section 6 - Full Data Cleaning Pipeline

This notebook cleans Online Retail data, engineers features, validates output, and saves a processed dataset for downstream analytics.

In [ ]:
# ================================
# 1. Imports and configuration
# ================================
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

In [ ]:
# ================================
# 2. Load dataset
# ================================
input_path = Path('../data/raw/online_retail.csv')
output_dir = Path('../data/processed')
output_path = output_dir / 'cleaned_data.csv'

# Proper encoding for this dataset
df = pd.read_csv(input_path, encoding='ISO-8859-1')

print(f'Loaded shape: {df.shape}')

In [ ]:
# ================================
# 3. Initial exploration
# ================================
print('Head of raw data:')
display(df.head())

print('Raw data info:')
df.info()

print('Missing values by column:')
display(df.isna().sum().sort_values(ascending=False))

In [ ]:
# ================================
# 4. Data cleaning
# ================================
rows_before = len(df)

# Handle missing values in CustomerID and Description
df = df.dropna(subset=['CustomerID', 'Description'])
rows_after_missing = len(df)

# Remove returns (negative quantity rows)
df = df[df['Quantity'] >= 0]
rows_after_quantity = len(df)

# Remove duplicate rows
df = df.drop_duplicates()
rows_after_dedup = len(df)

print(f'Rows before cleaning: {rows_before}')
print(f'Rows after dropping missing CustomerID/Description: {rows_after_missing}')
print(f'Rows after removing negative quantities: {rows_after_quantity}')
print(f'Rows after duplicate removal: {rows_after_dedup}')

In [ ]:
# ================================
# 5. Convert data types
# ================================
# Convert transaction date to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Drop rows where conversion failed
df = df.dropna(subset=['InvoiceDate'])

# Convert CustomerID to nullable integer type
df['CustomerID'] = df['CustomerID'].astype('Int64')

# Ensure identifier and country columns are strings
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)
df['Country'] = df['Country'].astype(str)

In [ ]:
# ================================
# 6. Feature engineering
# ================================
# Revenue feature
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Date-based features
df['InvoiceMonth'] = df['InvoiceDate'].dt.month
df['InvoiceYear'] = df['InvoiceDate'].dt.year

display(df[['InvoiceDate', 'InvoiceMonth', 'InvoiceYear', 'Quantity', 'UnitPrice', 'Revenue']].head())

In [ ]:
# ================================
# 7. Final validation
# ================================
print(f'Final dataset shape: {df.shape}')
print('Final dataset info:')
df.info()

In [ ]:
# ================================
# 8. Save cleaned dataset
# ================================
output_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print(f'Clean dataset saved to: {output_path.resolve()}')